# square_map: chess squares <-> world coordinates

Library notebook for `gambit_arm`. Converts a square name (`"e4"`) to the world point the
gripper must reach, and back (nearest square to an (x, y) point, e.g. from the camera).
Everything comes from `gambit_bringup/config/board.yaml`, so measuring the real board and
editing that file is all it takes to move from Gazebo to the real table.

Milestone: *Hardware and execution* (board position registered; pick-and-place needs targets).

**TODO**
- [x] load board.yaml (searching up from this folder, works in Colab after `%cd`)
- [x] square -> world (x, y, z) with the docs/gazebo.md formula, incl. `white_near_arm: false`
- [x] world (x, y) -> nearest square, with an outside-the-board check
- [x] graveyard slots (row along +x, same spacing as the Gazebo world)
- [x] piece heights and grasp heights from `pieces.height` and `arm.grasp_height_frac`
- [ ] after registering the real board (ArUco corners), replace `board.center` with the
      measured value and add the measured yaw if the board is rotated (formula below assumes 0)
- [ ] decide how many graveyard slots fit within reach (a second row?) once IK is validated

In [ ]:
from pathlib import Path
import math
import numpy as np
import yaml

## Finding the repo and loading board.yaml

Notebooks run with their own folder as the working directory (that is how `%run` paths
resolve), so walk upward until we find `gambit_bringup/config`.

In [ ]:
def find_repo_root(start=None):
    """Return the repo root (the folder containing gambit_bringup/config)."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "gambit_bringup" / "config").is_dir():
            return cand
    raise FileNotFoundError("could not find gambit_bringup/config above " + str(p))

REPO_ROOT = find_repo_root()
CONFIG_DIR = REPO_ROOT / "gambit_bringup" / "config"

def load_yaml(name):
    with open(CONFIG_DIR / name) as f:
        return yaml.safe_load(f)

BOARD = load_yaml("board.yaml")
BOARD["board"]

## Square -> world

From docs/gazebo.md: with file 0 = a and rank 0 = rank 1,

$$x = c_x + (\text{rank} - 3.5)\,s, \qquad y = c_y + (3.5 - \text{file})\,s, \qquad z = \text{thickness}$$

Rank grows away from the arm (+x); file a is on the arm's left (+y). If the board is turned
around (`white_near_arm: false`) both indices flip. `z` is the board top; the grasp height
is added on top of it per piece.

In [ ]:
FILES = "abcdefgh"

def parse_square(name):
    """'e4' -> (file=4, rank=3). Also accepts a python-chess index 0..63."""
    if isinstance(name, (int, np.integer)):
        if not 0 <= name < 64:
            raise ValueError(f"square index out of range: {name}")
        return int(name) % 8, int(name) // 8
    if len(name) != 2 or name[0] not in FILES or name[1] not in "12345678":
        raise ValueError(f"bad square name: {name!r}")
    return FILES.index(name[0]), int(name[1]) - 1

def square_name(file, rank):
    return f"{FILES[file]}{rank + 1}"

def square_to_world(name, cfg=BOARD):
    """Square center on the board top, world frame, meters -> np.array([x, y, z])."""
    f, r = parse_square(name)
    b = cfg["board"]
    s = b["square"]
    cx, cy = b["center"]
    if not b.get("white_near_arm", True):
        f, r = 7 - f, 7 - r
    return np.array([cx + (r - 3.5) * s, cy + (3.5 - f) * s, b["thickness"]])

## World -> nearest square

Invert the formula, round to the nearest index, and reject points outside the 8x8 playing
area (more than half a square beyond the outer centers).

In [ ]:
def world_to_square(x, y, cfg=BOARD):
    """Nearest square name to world (x, y), or None if outside the playing area."""
    b = cfg["board"]
    s = b["square"]
    cx, cy = b["center"]
    r = (x - cx) / s + 3.5
    f = 3.5 - (y - cy) / s
    if not (-0.5 <= r < 7.5 and -0.5 <= f < 7.5):
        return None
    r, f = min(7, max(0, round(r))), min(7, max(0, round(f)))
    if not b.get("white_near_arm", True):
        f, r = 7 - f, 7 - r
    return square_name(f, r)

def offset_from_center(x, y, cfg=BOARD):
    """(square, dx, dy): nearest square and the offset of (x, y) from its center (m).
    Used to check that a placed piece is centered within tolerance."""
    sq = world_to_square(x, y, cfg)
    if sq is None:
        return None, math.nan, math.nan
    c = square_to_world(sq, cfg)
    return sq, x - c[0], y - c[1]

## Graveyard and piece heights

Captured pieces go to a row along +x starting at `pieces.graveyard`, on the table (z = 0).
The spacing (0.9 square) matches `GRAVEYARD_SPACING` in gambit_sim/notebooks/world_generator.ipynb
so the Gazebo world's spare queens sit exactly on slots 0..3.

Grasp height = board top (or table) + `grasp_height_frac` * piece height.

In [ ]:
GRAVEYARD_SPACING_FRAC = 0.9

def graveyard_slot(index, cfg=BOARD):
    """World point of graveyard slot `index` (0, 1, ...) on the table top."""
    gx, gy = cfg["pieces"]["graveyard"]
    return np.array([gx + index * GRAVEYARD_SPACING_FRAC * cfg["board"]["square"], gy, 0.0])

def piece_height(kind, cfg=BOARD):
    """Piece height in m for kind in 'PNBRQK' (either case)."""
    return cfg["pieces"]["height"][kind.upper()] * cfg["board"]["square"]

def grasp_point(target, kind, cfg=BOARD):
    """Where the gripper tip should be to grasp piece `kind` standing at `target`
    (a square name or a world point such as a graveyard slot)."""
    base = square_to_world(target, cfg) if isinstance(target, (str, int)) else np.asarray(target, float)
    return base + np.array([0, 0, cfg["arm"]["grasp_height_frac"] * piece_height(kind, cfg)])

def hover_point(target, cfg=BOARD):
    """Point `arm.hover_height` above the board top (or table) at the target."""
    base = square_to_world(target, cfg) if isinstance(target, (str, int)) else np.asarray(target, float)
    return base + np.array([0, 0, cfg["arm"]["hover_height"]])

ALL_SQUARES = [square_name(f, r) for r in range(8) for f in range(8)]

## Self-check

In [ ]:
# docs/gazebo.md example: e2 pawn teleported to e4 at (0.2025, -0.0125, 0.006)
assert np.allclose(square_to_world("e4"), [0.2025, -0.0125, 0.006])
assert np.allclose(square_to_world("e2"), [0.1525, -0.0125, 0.006])
assert np.allclose(square_to_world("a1")[:2], [0.1275, 0.0875])
assert np.allclose(square_to_world("h8")[:2], [0.3025, -0.0875])
assert np.allclose(square_to_world(28), square_to_world("e4"))      # python-chess index
for sq in ALL_SQUARES:
    x, y, _ = square_to_world(sq)
    assert world_to_square(x + 0.01, y - 0.01) == sq
assert world_to_square(0.0, 0.0) is None
sq, dx, dy = offset_from_center(0.2045, -0.0125)
assert sq == "e4" and abs(dx - 0.002) < 1e-9 and abs(dy) < 1e-9
_rot = {**BOARD, "board": {**BOARD["board"], "white_near_arm": False}}
assert np.allclose(square_to_world("h8", _rot)[:2], [0.1275, 0.0875])
assert world_to_square(0.1275, 0.0875, _rot) == "h8"
# spare queens in chess_world.sdf sit at x = 0.17, 0.1925, 0.215, 0.2375, y = -0.175
assert np.allclose([graveyard_slot(i)[0] for i in range(4)], [0.17, 0.1925, 0.215, 0.2375])
assert np.isclose(grasp_point("e4", "P")[2], 0.006 + 0.55 * 0.9 * 0.025)
assert np.isclose(hover_point("e4")[2], 0.076)
print("square_map OK")